In [2]:
import sys
import os

import jax
import jax.numpy as np
import jax.random as jr
import equinox as eqx

import amigo as amg
import dorito as drt

from kneed import KneeLocator

# matplotlib configs
from matplotlib import pyplot as plt
import matplotlib as mpl
import ehtplot
import scienceplots

plt.style.use(['science',  'bright', 'no-latex'])
new_rcParams = {
    'image.cmap': 'inferno',
    'font.family': 'serif',
    'image.origin': 'lower',
    'figure.dpi': 300,
    'font.size': 8,
    'xtick.direction': 'out',
    'ytick.direction': 'out'
}
plt.rcParams.update(new_rcParams)

inferno = mpl.colormaps['inferno']
viridis = mpl.colormaps['viridis']
seismic = mpl.colormaps['seismic']
coolwarm = mpl.colormaps['coolwarm']

inferno.set_bad('k', 0.5)
viridis.set_bad('k', 0.5)
seismic.set_bad('k', 0.5)
coolwarm.set_bad('k', 0.5)

from frito.autoencoder.ae_utils import load_classes_from_file as lcf

In [ ]:
master_key              = jr.key(0)
main_data_path          = '_data'
autoencoder_data_path   = os.path.join(main_data_path, 'autoencoder')
main_trained_model_path = os.path.join(autoencoder_data_path, 'trained_models')
main_svd_path           = os.path.join(autoencoder_data_path, 'svd')
training_data_path      = os.path.join(autoencoder_data_path, 'training_data')
emnist_path             = os.path.join(training_data_path, 'emnist.npz')
mnist_path              = os.path.join(training_data_path, 'mnist.npz')
ppd_51_path             = os.path.join(training_data_path, 'fake_intensity_PPDs_51x51.npz')

main_out_path           = '_output'
main_model_struct_path  = 'src/frito/autoencoder/model_structures'

ppd_51x51_data          = np.load(ppd_51_path)
test_51x51_data         = ppd_51x51_data['x_test']

disco_HD135344B_path = os.path.join(main_data_path, 'jwst', 'HD135344B', 'disco', 'cal_vis_HD135344B.npy')

In [11]:
disco_HD135344B = np.load(disco_HD135344B_path, allow_pickle=True)

In [14]:
disco_HD135344B

array({'F380M': {'u': Array([-8.206427  , -8.123488  , -8.04055   , ..., -0.24881592,
       -0.16587728, -0.08293864], dtype=float32), 'v': Array([-4.059495  , -3.8141766 , -3.5688581 , ..., -0.73595536,
       -0.4906369 , -0.24531844], dtype=float32), 'vis_mat': Array([[ 8.5335134e-14,  4.2682524e-13,  8.7069241e-13, ...,
         9.1124320e-04,  9.0259746e-02,  6.2886512e-01],
       [-3.5199824e-13,  1.2659318e-13,  1.2889689e-13, ...,
         2.2056140e-03,  1.4687042e-01,  6.8538535e-01],
       [ 4.7325891e-13, -1.3889445e-12, -4.5785598e-13, ...,
         2.5401857e-06,  3.7881266e-05,  4.7295736e-04],
       ...,
       [ 7.3883184e-11,  1.4862811e-10,  2.0596574e-10, ...,
         1.4768610e-05, -2.0104082e-08,  5.7770134e-08],
       [ 6.7626920e-12,  7.7640144e-11,  6.4228491e-11, ...,
        -1.4719413e-06,  1.2680597e-07, -3.6954393e-08],
       [-6.8767526e-11, -1.6118809e-10, -1.4584332e-10, ...,
         3.5851463e-04, -3.3284796e-05,  4.9554487e-06]], dtype=float32